In [154]:
import numpy as np
import pandas as pd
import math
from scipy.special import exp1
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [155]:
####################
### domain setup ###
####################

# number of wells
n_wells = 3  

# time steps for pumping rate changes
t_eval_val = 365.0 # [day]
dt = 7 # [days]
time_steps = np.arange(dt, t_eval_val + dt, dt)
if time_steps[-1] > t_eval_val: time_steps[-1] = t_eval_val

# bounds of coordinate system
x_min, x_max = -50, 50
y_min, y_max = -50, 50

# wells cannot be within this many units of the centre (0,0)
no_go_radius = 10.0

# range of pumping rates [m3/day]
q_min, q_max = 6.5, 65

# aquifer properties
T_val = 100.0  # [m2/day]
S_val = 0.001  # [-]

##############################
### random well generation ###
##############################

wells = []
while len(wells) < n_wells:
    wx, wy = np.random.uniform(x_min, x_max), np.random.uniform(y_min, y_max)
    # check distance for all wells from centre
    if np.sqrt(wx**2 + wy**2) > no_go_radius:
        wells.append({
            'X': wx, 'Y': wy, 
            'Q_series': np.random.uniform(q_min, q_max, len(time_steps))
        })

# randomly select yc as either y_min or y_max to place the compliance point on the top or bottom boundary
xc_val = np.random.uniform(x_min, x_max)
yc = np.random.choice([y_min, y_max])
r_centre_to_cp = np.sqrt(xc_val**2 + yc**2)

def get_s_at_t(q_series, times, target_t, r, T, S):
    """Calculates drawdown at time target_t using temporal superposition"""
    if r < 1e-3: return 0
    # impact of initial pumping rate at t=0
    u0 = (r**2 * S) / (4.0 * T * target_t)
    s = (q_series[0] / (4.0 * math.pi * T)) * exp1(u0)
    
    # impact of subsequent pumping rate changes
    for i in range(1, len(times)):
        if times[i-1] >= target_t: break
        dq = q_series[i] - q_series[i-1]
        u = (r**2 * S) / (4.0 * T * (target_t - times[i-1]))
        s += (dq / (4.0 * math.pi * T)) * exp1(u)
    return s

################
### solution ###
################

# calculate drawdown at compliance point for each time step from all random wells
s_target_series = []
for t in time_steps:
    s_t = sum(get_s_at_t(w['Q_series'], time_steps, t, 
                         np.sqrt((w['X']-xc_val)**2 + (w['Y']-yc)**2), 
                         T_val, S_val) for w in wells)
    s_target_series.append(s_t)

# calculate equivalent pumping rate at centre well needed to match the target drawdown at each time step 
# using temporal superposition
q_equiv_series = []
for i, t in enumerate(time_steps):
    target_s = s_target_series[i]
    if i == 0:
        u = (r_centre_to_cp**2 * S_val) / (4.0 * T_val * t)
        q_needed = (target_s * 4.0 * math.pi * T_val) / exp1(u)
    else:
        # calculate drawdown at compliance point from previous pumping rates at centre well
        s_prev_impact = (q_equiv_series[0] / (4.0 * math.pi * T_val)) * exp1((r_centre_to_cp**2 * S_val) / (4.0 * T_val * t))
        for j in range(1, i):
            dq = q_equiv_series[j] - q_equiv_series[j-1]
            s_prev_impact += (dq / (4.0 * math.pi * T_val)) * exp1((r_centre_to_cp**2 * S_val) / (4.0 * T_val * (t - time_steps[j-1])))
        
        # calculate additional pumping needed at centre to make up the difference to target drawdown
        u_now = (r_centre_to_cp**2 * S_val) / (4.0 * T_val * (t - time_steps[i-1]))
        dq_needed = (target_s - s_prev_impact) * (4.0 * math.pi * T_val) / exp1(u_now)
        q_needed = q_equiv_series[i-1] + dq_needed
        
    q_equiv_series.append(q_needed)


In [ ]:
#####################
### visualisation ###
#####################

# calculate individual drawdown series for each well for plotting
individual_s_series = []
for w in wells:
    r = np.sqrt((w['X'] - xc_val)**2 + (w['Y'] - yc)**2)
    s_w = [get_s_at_t(w['Q_series'], time_steps, t, r, T_val, S_val) for t in time_steps]
    individual_s_series.append(s_w)

# create subplots with shared x-axis
fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.1,
    subplot_titles=("Weekly Drawdown at Compliance Point", "Weekly Pumping Rates")
)

# colour palette for individual wells
colours = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A']

# first row: drawdown at compliance point
# individual random well drawdowns
for i, s_w in enumerate(individual_s_series):
    fig.add_trace(go.Scatter(
        x=time_steps, y=s_w, 
        name=f"Well {i+1}", 
        line=dict(color=colours[i % len(colours)], width=0.5),
        legendgroup=f"well{i+1}"
    ), row=1, col=1)

# superimpose total drawdown from all random wells at compliance point
fig.add_trace(go.Scatter(
    x=time_steps, y=s_target_series, 
    name=f"Superimposition of {len(wells)} Wells Drawdown (Target)", 
    line=dict(color='green', width=2)
), row=1, col=1)

# centre well match
fig.add_trace(go.Scatter(
    x=time_steps, y=s_target_series, 
    name=f"Equivalent Centre Well (Match)", 
    line=dict(color='red', dash='dot', width=2)
), row=1, col=1)

# second row: pumping rates
# individual random well pumping rates
for i, w in enumerate(wells):
    fig.add_trace(go.Scatter(
        x=time_steps, y=w['Q_series'], 
        name=f"Well {i+1} Q", 
        line=dict(color=colours[i % len(colours)], width=0.5),
        legendgroup=f"well{i+1}",
        showlegend=False
    ), row=2, col=1)

# equivalent centre well pumping rate
fig.add_trace(go.Scatter(
    x=time_steps, y=q_equiv_series, 
    name="Centre Well Pumping Rate", 
    line=dict(color='black', width=2)
), row=2, col=1)

# pretty up the layout
fig.update_layout(
    height=800, 
    width=900, 
    template="plotly_white",
    title_text=f"Multi-Well Theis Analysis: {len(wells)} Individual Wells vs. Equivalent Centre Well Responses",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.05)
)

fig.update_yaxes(title_text="Drawdown (m)", row=1, col=1)
fig.update_yaxes(title_text="Pumping Rate (m³/d)", row=2, col=1)
fig.update_xaxes(title_text="Days", row=2, col=1)

fig.show()

In [157]:
###################
### monte carlo ###
###################

####################
### domain setup ###
####################

# number of wells
n_wells = 3  

# time steps for pumping rate changes
t_eval_val = 365.0 # [day]
dt = 7 # [days]
time_steps = np.arange(dt, t_eval_val + dt, dt)
if time_steps[-1] > t_eval_val: time_steps[-1] = t_eval_val
n_steps = len(time_steps)

# bounds of coordinate system
x_min, x_max = -50, 50
y_min, y_max = -50, 50

# wells cannot be within this many units of the centre (0,0)
no_go_radius = 10.0

# range of pumping rates [m3/day]
q_min, q_max = 6.5, 65

# aquifer properties
T_val = 100.0  # [m2/day]
S_val = 0.001  # [-]

def get_s_at_t(q_series, times, target_t, r, T, S):
    """Calculates drawdown at time target_t using temporal superposition"""
    if r < 1e-3: return 0
    u0 = (r**2 * S) / (4.0 * T * target_t)
    s = (q_series[0] / (4.0 * math.pi * T)) * exp1(u0)
    for i in range(1, len(times)):
        if times[i-1] >= target_t: break
        dq = q_series[i] - q_series[i-1]
        u = (r**2 * S) / (4.0 * T * (target_t - times[i-1]))
        s += (dq / (4.0 * math.pi * T)) * exp1(u)
    return s

def solve_centre_well(s_targets, r_centre, times, T, S):
    """Solves for the required centre pumping series step-by-step"""
    q_out = []
    for i, t in enumerate(times):
        if i == 0:
            u = (r_centre**2 * S) / (4.0 * T * t)
            q_needed = (s_targets[0] * 4.0 * math.pi * T) / exp1(u)
        else:
            # Impact of historical rates
            u_init = (r_centre**2 * S) / (4.0 * T * t)
            s_prev_impact = (q_out[0] / (4.0 * math.pi * T)) * exp1(u_init)
            for j in range(1, i):
                dq = q_out[j] - q_out[j-1]
                u_hist = (r_centre**2 * S) / (4.0 * T * (t - times[j-1]))
                s_prev_impact += (dq / (4.0 * math.pi * T)) * exp1(u_hist)
            # Find current step Q
            u_now = (r_centre**2 * S) / (4.0 * T * (t - times[i-1]))
            dq_needed = (s_targets[i] - s_prev_impact) * (4.0 * math.pi * T) / exp1(u_now)
            q_needed = q_out[-1] + dq_needed
        q_out.append(q_needed)
    return q_out

#########################
### monte carlo setup ###
#########################

num_reals = 100

fig = go.Figure()

for m in range(num_reals):
    # randomly select compliance point on top or bottom boundary and calculate centre distance
    xc, yc = np.random.uniform(x_min, x_max), np.random.choice([y_min, y_max])
    r_centre = np.sqrt(xc**2 + yc**2)
    
    # generate random wells with random pumping rates
    m_wells = []
    for _ in range(n_wells):
        while True:
            wx, wy = np.random.uniform(x_min, x_max), np.random.uniform(y_min, y_max)
            if np.sqrt(wx**2 + wy**2) > no_go_radius:
                m_wells.append({'r': np.sqrt((wx-xc)**2 + (wy-yc)**2), 'Q': np.random.uniform(q_min, q_max, n_steps)})
                break
    
    # calculate target drawdown at compliance point from all random wells
    s_target = [sum(get_s_at_t(w['Q'], time_steps, t, w['r'], T_val, S_val) for w in m_wells) for t in time_steps]
    
    # match using centre well
    q_equiv = solve_centre_well(s_target, r_centre, time_steps, T_val, S_val)
    s_centre = [get_s_at_t(q_equiv, time_steps, t, r_centre, T_val, S_val) for t in time_steps]

    # plot
    show_leg = True if m == 0 else False
    fig.add_trace(go.Scatter(x=time_steps, y=s_target, mode='lines', 
                             line=dict(color='red', width=0.5), opacity=0.5, 
                             name=f"Superposition of {len(m_wells)} Random Wells (Target)", showlegend=show_leg))
    fig.add_trace(go.Scatter(x=time_steps, y=s_centre, mode='lines', 
                             line=dict(color='blue', width=0.5), opacity=0.5, 
                             name=f"Equivalent Centre Well (Match)", showlegend=show_leg))

# pretty up the layout
fig.update_layout(
    title=f"Monte Carlo Verification: {num_reals} Realisations",
    xaxis_title="Days", yaxis_title="Drawdown (m)",
    template="plotly_white",
    hovermode=False # Performance boost for 2000 lines
)
fig.show()
